# 02. Nexus 축소판 만들기

목표: Nexus의 멀티 에이전트 구조를 LLM 없이 Python 함수로 흉내 낸다.

실행 방법:
1. `01_foundations.ipynb`를 먼저 실행해 기본 용어를 익힌다.
2. 필요한 패키지가 없다면 `pip install -r requirements.txt`를 실행한다.
3. 이 노트북을 위에서 아래로 실행한다.
4. 각 함수가 Nexus의 어떤 에이전트 역할에 대응하는지 주석을 확인한다.

여기서 agent는 자율 시스템이 아니라 특정 책임을 맡은 함수로 표현한다. 실제 논문에서는 이 역할을 LLM 호출과 프롬프트가 수행한다.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)

## 1. 사건이 포함된 데이터 준비

미래 이벤트를 완전히 안다고 가정하면 현실적이지 않지만, 실습에서는 에이전트 구조를 이해하는 것이 목적이다. 실제 적용에서는 미래 이벤트 대신 알려진 일정, 시나리오, 외부 전망을 넣는다.

In [ ]:
def make_dataset(n=96):
    weeks = np.arange(1, n + 1)
    trend = 80 + weeks * 0.55
    seasonality = 6 * np.sin(2 * np.pi * weeks / 12)
    event_effect = np.zeros(n)
    events = []

    for i, week in enumerate(weeks):
        if 35 <= week <= 42:
            event_effect[i] += 10
            events.append("supply shortage lifts prices")
        elif 70 <= week <= 78:
            event_effect[i] -= 12
            events.append("new policy cools demand")
        else:
            events.append("normal conditions")

    noise = rng.normal(0, 1.6, n)
    value = trend + seasonality + event_effect + noise
    return pd.DataFrame({"week": weeks, "value": value, "event": events})


df = make_dataset()
train = df.iloc[:-16].copy()
future = df.iloc[-16:].copy()
df.tail()

## 2. Historical Context Agent

이 에이전트는 원시 데이터를 시점별 설명 구조로 바꾼다. 중요한 점은 수치 변화와 사건 텍스트를 같은 행에 묶는 것이다.

In [ ]:
def classify_event(event_text):
    """텍스트를 방향성 있는 이벤트 신호로 바꾼다.

    실제 Nexus의 LLM 에이전트는 더 풍부한 자연어 이해를 수행한다.
    여기서는 구조를 배우기 위해 키워드 기반 규칙을 사용한다.
    """
    text = event_text.lower()
    if "shortage" in text or "lifts" in text:
        return "positive", 1.0
    if "cools" in text or "policy" in text:
        return "negative", -1.0
    return "neutral", 0.0


def historical_context_agent(frame):
    """수치와 사건을 연결한 구조화 타임라인을 만든다."""
    rows = []
    prev_value = None
    for row in frame.itertuples(index=False):
        change = 0.0 if prev_value is None else float(row.value - prev_value)
        label, signal = classify_event(row.event)
        rows.append({
            "week": int(row.week),
            "value": float(row.value),
            "change": change,
            "event": row.event,
            "event_label": label,
            "event_signal": signal,
            "brief": f"week {int(row.week)}: value={row.value:.1f}, change={change:+.1f}, event={label}",
        })
        prev_value = row.value
    return pd.DataFrame(rows)


context = historical_context_agent(train)
context.tail(8)

## 3. Macro-Reasoning Agent

거시 에이전트는 전체 흐름을 본다. 여기서는 선형 추세와 계절 평균을 사용한다. 복잡한 모델을 쓰지 않는 이유는 역할 분해 자체를 보기 위해서다.

In [ ]:
def macro_reasoning_agent(context, future_weeks):
    """장기 추세와 계절 패턴으로 넓은 궤적을 만든다."""
    x = context["week"].to_numpy()
    y = context["value"].to_numpy()

    # 1차 회귀는 장기 추세를 간단하게 잡는 방법이다.
    slope, intercept = np.polyfit(x, y, deg=1)
    trend_forecast = intercept + slope * np.asarray(future_weeks)

    # 같은 계절 위치의 평균 잔차를 더해 반복 패턴을 반영한다.
    residual = y - (intercept + slope * x)
    season_key = ((context["week"] - 1) % 12).to_numpy()
    season_table = pd.DataFrame({"season": season_key, "residual": residual}).groupby("season").mean()
    seasonal = np.array([season_table.loc[(w - 1) % 12, "residual"] for w in future_weeks])

    forecast = trend_forecast + seasonal
    rationale = f"macro: slope={slope:.3f}; long-run trend plus 12-week seasonal residuals"
    return forecast, rationale


macro_pred, macro_rationale = macro_reasoning_agent(context, future["week"].to_numpy())
macro_rationale

## 4. Micro-Reasoning Agent

미시 에이전트는 최근 변화와 미래 이벤트의 짧은 영향에 집중한다. 이 예제에서는 최근 평균 변화량과 이벤트 신호를 더한다.

In [ ]:
def micro_reasoning_agent(context, future_events):
    """단기 변화율과 이벤트 촉매로 한 시점씩 예측한다."""
    recent_changes = context["change"].tail(6).to_numpy()
    recent_drift = float(np.mean(recent_changes))
    current = float(context["value"].iloc[-1])

    predictions = []
    rationales = []
    for step, event_text in enumerate(future_events, start=1):
        label, signal = classify_event(event_text)
        # 이벤트 효과가 시간이 지나며 조금씩 약해진다고 가정한다.
        event_bump = signal * 7.0 * np.exp(-(step - 1) / 6)
        current = current + recent_drift + event_bump
        predictions.append(current)
        rationales.append(f"t+{step}: recent_drift={recent_drift:+.2f}, event={label}, bump={event_bump:+.2f}")

    return np.array(predictions), rationales


micro_pred, micro_rationales = micro_reasoning_agent(context, future["event"].tolist())
micro_rationales[:3]

## 5. Forecast Synthesizer Agent

종합 에이전트는 거시와 미시 예측을 결합한다. 변동성이 큰 이벤트 구간에서는 미시 관점을 조금 더 믿고, 일반 구간에서는 거시 관점을 더 믿도록 가중치를 바꾼다.

In [ ]:
def forecast_synthesizer_agent(macro_pred, macro_rationale, micro_pred, micro_rationales, future_events):
    """두 전망을 하나의 최종 예측과 설명으로 결합한다."""
    final = []
    explanations = []
    for i, event_text in enumerate(future_events):
        label, signal = classify_event(event_text)
        # 이벤트가 강하면 미시 예측의 가중치를 높인다.
        micro_weight = 0.62 if signal != 0 else 0.42
        macro_weight = 1.0 - micro_weight
        value = macro_weight * macro_pred[i] + micro_weight * micro_pred[i]
        final.append(value)
        explanations.append(
            f"week {i + 1}: macro_weight={macro_weight:.2f}, micro_weight={micro_weight:.2f}; "
            f"event={label}; {micro_rationales[i]}"
        )
    return np.array(final), explanations


final_pred, explanations = forecast_synthesizer_agent(
    macro_pred,
    macro_rationale,
    micro_pred,
    micro_rationales,
    future["event"].tolist(),
)
explanations[:3]

## 6. 기준선과 비교

멀티 에이전트 구조가 항상 이기는 것은 아니다. 그래서 naive, macro-only, micro-only와 함께 비교해야 한다.

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return math.sqrt(np.mean((y_true - y_pred) ** 2))


actual = future["value"].to_numpy()
naive = np.repeat(float(train["value"].iloc[-1]), len(future))

comparison = pd.DataFrame([
    {"model": "naive", "MAPE": mape(actual, naive), "RMSE": rmse(actual, naive)},
    {"model": "macro_only", "MAPE": mape(actual, macro_pred), "RMSE": rmse(actual, macro_pred)},
    {"model": "micro_only", "MAPE": mape(actual, micro_pred), "RMSE": rmse(actual, micro_pred)},
    {"model": "synthesized", "MAPE": mape(actual, final_pred), "RMSE": rmse(actual, final_pred)},
])
comparison.sort_values("MAPE")

In [ ]:
plot_df = future[["week", "value"]].copy()
plot_df["macro"] = macro_pred
plot_df["micro"] = micro_pred
plot_df["synthesized"] = final_pred

ax = plot_df.plot(x="week", y=["value", "macro", "micro", "synthesized"], figsize=(11, 4))
ax.set_title("Macro, micro, and synthesized forecasts")
ax.set_ylabel("value")
plt.show()

## 정리

- Historical Context Agent는 원시 입력을 구조화한다.
- Macro Agent는 큰 흐름과 계절성을 담당한다.
- Micro Agent는 이벤트와 최근 변화를 담당한다.
- Synthesizer는 어떤 관점을 더 믿을지 결정하고 설명을 남긴다.
- 다음 노트북에서는 과거 백테스트로 이 결합 규칙을 보정한다.